# 4.3 · 多项式回归 / Polynomial Regression

> **课程定位 / Where this fits**
> **Part 4 第 3 课**。4.2 的残差曲率说明线性不够; 多项式回归是**第一招非线性**——而且它**仍是线性模型**（对参数线性）。本课同时是**偏差-方差权衡 / 欠拟合-过拟合**的最佳教学案例。
> The first nonlinearity, yet still a linear model (in the parameters). Also the best teaching case for bias-variance / under-vs-overfitting.

> 💡 **面试相关 / Interview-relevant**
> - "多项式回归为什么还算线性模型" ★★★★（对参数线性）
> - "怎么判断过拟合/欠拟合" ★★★★★
> - "偏差-方差权衡" ★★★★★（0.10 见过, 这里可视化）
> - "多项式阶数怎么选" ★★★（CV）

---

## 学习目标 / Learning Objectives
1. 理解**基函数展开**：$x \to [1, x, x^2, \dots]$ 后仍是线性回归。
2. 亲眼看到**欠拟合 / 恰好 / 过拟合**随阶数的变化。
3. 用**学习曲线 + 验证曲线**诊断偏差-方差。
4. 用 CV 选最优阶数, 并理解正则化（4.4 预告）是另一条路。

## 目录 / TOC
1. [基函数展开: 非线性的线性模型 ⭐](#1)
2. [数据: 真实曲线 + 噪声](#2)
3. [欠拟合→恰好→过拟合 ⭐](#3)
4. [偏差-方差权衡可视化 ⭐](#4)
5. [验证曲线: 选阶数](#5)
6. [学习曲线: 诊断](#6)
7. [小结](#7)


<a id="1"></a>
## 1. 基函数展开: 非线性的线性模型 ⭐ / Basis Expansion

**核心洞察**: 把单特征 $x$ 扩展成 $[1, x, x^2, \dots, x^p]$, 再做**普通线性回归**：
$$\hat{y} = w_0 + w_1 x + w_2 x^2 + \dots + w_p x^p$$

虽然 $\hat{y}$ 关于 $x$ 是**非线性**的, 但关于**参数 $\mathbf{w}$ 仍是线性**——所以 4.1 的全套（正规方程、最小二乘）**原封不动适用**。这叫**基函数展开**, 是核方法(4.9)、样条等一切"用线性工具拟合非线性"的统一思想。

| 关于什么线性 | 多项式回归 |
|---|---|
| 关于 $x$ | ❌ 非线性（曲线）|
| 关于参数 $\mathbf{w}$ | ✅ 线性（所以叫"线性模型"）|

**"线性模型"的"线性"指的是参数, 不是特征**——这是最常被误解的术语点。
"Linear" in "linear model" refers to the parameters, not the features — the most misunderstood term.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, validation_curve, learning_curve
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# 真实函数: 三次曲线 + 噪声 / true cubic + noise
def true_f(x): return 0.5*x**3 - 2*x**2 + x + 3
n = 80
x = np.sort(rng.uniform(-2, 4, n))
y = true_f(x) + rng.normal(0, 3, n)
X = x.reshape(-1, 1)
print(f"真实函数: 0.5x³-2x²+x+3 (3阶), 加噪声 σ=3, n={n}")


<a id="3"></a>
## 3. 欠拟合 → 恰好 → 过拟合 ⭐ / Under → Just-right → Over

同一份数据, 拟合不同阶数的多项式——**经典三态**。


In [ ]:
x_plot = np.linspace(-2, 4, 300).reshape(-1, 1)
degrees = [1, 3, 15]
titles = ["欠拟合 (阶=1, 直线拟合曲线)", "恰好 (阶=3, 匹配真实)", "过拟合 (阶=15, 追噪声)"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, deg, title in zip(axes, degrees, titles):
    model = make_pipeline(PolynomialFeatures(deg), StandardScaler(), LinearRegression())
    model.fit(X, y)
    train_r2 = model.score(X, y)
    cv_r2 = cross_val_score(model, X, y, cv=5, scoring="r2").mean()
    ax.scatter(x, y, alpha=0.4, s=15)
    ax.plot(x_plot, true_f(x_plot), "g--", lw=1.5, label="真实函数")
    ax.plot(x_plot, model.predict(x_plot), "r-", lw=2, label=f"拟合 (阶={deg})")
    ax.set_ylim(y.min()-5, y.max()+5)
    ax.set_title(f"{title}\ntrain R²={train_r2:.2f}, CV R²={cv_r2:.2f}")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print("欠拟合: train 和 CV 都差 (模型太简单, 高偏差)")
print("恰好: train 和 CV 都好且接近")
print("过拟合: train 极好但 CV 崩 (追逐噪声, 高方差) — train>>CV 是过拟合签名")


**三态诊断口诀**（面试必背）：
- **欠拟合**：train 差, CV 也差, 两者接近 → **高偏差** → 加复杂度
- **恰好**：train 好, CV 好, 两者接近
- **过拟合**：train 极好, **CV 差**, train >> CV → **高方差** → 减复杂度 / 加正则 / 加数据

`train >> test/CV` 是过拟合的签名（对比 3.9: 泄漏是 test 也虚高但生产崩）。


<a id="4"></a>
## 4. 偏差-方差权衡可视化 ⭐ / Bias-Variance Tradeoff

0.10 节理论提过, 这里**实测**。对每个阶数, 重复采样多次训练, 看预测的**偏差²**和**方差**如何此消彼长。

$$\mathbb{E}[(y - \hat{f})^2] = \underbrace{\text{Bias}^2}_{\text{模型太简单}} + \underbrace{\text{Var}}_{\text{模型太敏感}} + \underbrace{\sigma^2}_{\text{不可约噪声}}$$


In [ ]:
# 对每个阶数, 多次重采样训练, 在固定测试点上测 bias²/variance
x_test = np.linspace(-1.5, 3.5, 50).reshape(-1, 1)
y_test_true = true_f(x_test).ravel()
degrees_range = range(1, 13)
n_repeats = 100

bias2_list, var_list, total_list = [], [], []
for deg in degrees_range:
    preds = np.zeros((n_repeats, len(x_test)))
    for r in range(n_repeats):
        xr = rng.uniform(-2, 4, n)
        yr = true_f(xr) + rng.normal(0, 3, n)
        m = make_pipeline(PolynomialFeatures(deg), StandardScaler(), LinearRegression())
        m.fit(xr.reshape(-1,1), yr)
        preds[r] = m.predict(x_test)
    mean_pred = preds.mean(axis=0)
    bias2 = np.mean((mean_pred - y_test_true)**2)        # 平均预测偏离真值
    var = np.mean(preds.var(axis=0))                      # 预测的波动
    bias2_list.append(bias2); var_list.append(var); total_list.append(bias2+var)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(list(degrees_range), bias2_list, "o-", label="Bias² (偏差, 模型太简单)")
ax.plot(list(degrees_range), var_list, "s-", label="Variance (方差, 模型太敏感)")
ax.plot(list(degrees_range), total_list, "^-", lw=2, label="Bias²+Var (总误差)")
ax.axvline(3, color="g", ls="--", alpha=0.5, label="真实阶数=3")
ax.set_xlabel("多项式阶数 (模型复杂度)"); ax.set_ylabel("error")
ax.set_yscale("log"); ax.legend(); ax.set_title("偏差-方差权衡: 总误差在中等复杂度最低")
plt.tight_layout(); plt.show()
print("阶数低: 高偏差(欠拟合); 阶数高: 高方差(过拟合); 总误差 U 形, 最优在中间(≈真实阶数3)")


<a id="5"></a>
## 5. 验证曲线: 选阶数 / Validation Curve

`validation_curve` 直接画"超参 vs train/CV 分数"——**选阶数的标准工具**, 同时一眼看出偏差-方差。


In [ ]:
degrees_v = np.arange(1, 16)
train_scores, val_scores = validation_curve(
    make_pipeline(PolynomialFeatures(), StandardScaler(), LinearRegression()),
    X, y, param_name="polynomialfeatures__degree", param_range=degrees_v,
    cv=5, scoring="r2")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(degrees_v, train_scores.mean(axis=1), "o-", label="train R²")
ax.plot(degrees_v, val_scores.mean(axis=1), "s-", label="CV R²")
ax.fill_between(degrees_v, val_scores.mean(1)-val_scores.std(1),
                val_scores.mean(1)+val_scores.std(1), alpha=0.2)
best_deg = degrees_v[np.argmax(val_scores.mean(axis=1))]
ax.axvline(best_deg, color="r", ls="--", label=f"最优阶数={best_deg}")
ax.set_xlabel("degree"); ax.set_ylabel("R²"); ax.set_ylim(0, 1.05); ax.legend()
ax.set_title("验证曲线: train↑但 CV 先升后降, 选 CV 峰值")
plt.tight_layout(); plt.show()
print(f"CV 选出最优阶数 = {best_deg} (接近真实 3)")
print("注意: 阶数↑ → train R² 单调升(总能更贴训练点), 但 CV 先升后降 → 这就是过拟合的边界")


<a id="6"></a>
## 6. 学习曲线: 诊断 / Learning Curve

`learning_curve` 画"训练集大小 vs 分数"——**诊断该加数据还是改模型**：
- train 和 CV 间**大缺口**且都未饱和 → 高方差 → **加数据有用**
- train 和 CV **早早收敛到低位** → 高偏差 → **加数据没用, 要换更强模型**


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, deg, title in [(axes[0], 1, "高偏差 (阶=1, 欠拟合)"),
                        (axes[1], 15, "高方差 (阶=15, 过拟合)")]:
    sizes, tr, va = learning_curve(
        make_pipeline(PolynomialFeatures(deg), StandardScaler(), LinearRegression()),
        X, y, cv=5, scoring="r2", train_sizes=np.linspace(0.2, 1.0, 8))
    ax.plot(sizes, tr.mean(1), "o-", label="train")
    ax.plot(sizes, va.mean(1), "s-", label="CV")
    ax.set_xlabel("训练样本数"); ax.set_ylabel("R²"); ax.legend(); ax.set_title(title)
plt.tight_layout(); plt.show()
print("左(高偏差): train/CV 都低且收敛 → 加数据没用, 要更复杂模型")
print("右(高方差): train 高 CV 低有大缺口 → 加数据能缩小缺口 (或减复杂度/加正则)")


<a id="7"></a>
## 7. 小结 / Summary

```
多项式回归 = 基函数展开 [1,x,x²,...,xᵖ] + 线性回归
  "线性模型"的线性指参数, 非特征 ⭐
三态: 欠拟合(train差CV差) / 恰好 / 过拟合(train好CV崩, train>>CV)
偏差-方差: 总误差 = Bias²+Var+σ²; 复杂度↑则偏差↓方差↑, U形最优在中间
验证曲线: 超参 vs train/CV → 选 CV 峰值
学习曲线: 样本数 vs 分数 → 高方差加数据有用, 高偏差换模型
```

### 💡 面试速查
1. **多项式仍是线性模型**: 对参数线性
2. **过拟合签名**: train >> CV (对比泄漏: test 也虚高)
3. **偏差-方差**: 简单模型高偏差, 复杂模型高方差, U 形权衡
4. **学习曲线诊断**: 大缺口未饱和→加数据; 早收敛低位→换模型
5. **高阶过拟合的另一解**: 不降阶, 加正则化(4.4 Ridge)压制系数

### 下一节
**4.4 岭回归 Ridge**——过拟合除了降复杂度, 还能"留着高阶但惩罚大系数": L2 正则化, 也专治 4.2 的多重共线。
